# Hansen《Econometrics》第 3 章习题解答

**Chapter 3 The Algebra of Least Squares**

对应书稿 PDF 第 112–116 页（印刷页 92–96），§3.26 Exercises。

完整推导见 `Hansen_Ch03_Exercises_Solutions.md`。  
本 notebook：理论摘要 + **Exercise 3.24–3.26 计算**。


## Exercise 3.1–3.11 代数核心

| 题 | 结论 |
|:--:|------|
| 3.1 | 矩条件 $\Rightarrow$ 样本均值与 $1/n$ 方差 |
| 3.2 | $Z=XC$ 可逆 $\Rightarrow$ 残差不变 |
| 3.3 | $X'\hat e=0$ |
| 3.4 | $X_2'\hat e=0$ |
| 3.5 | $\hat e$ 对 $X$ 回归系数 $=0$ |
| 3.6 | $\hat Y$ 对 $X$ 回归系数 $=\hat\beta$ |
| 3.7–3.10 | 投影/幂等/迹/正交分块 |
| 3.11 | 含截距 $\Rightarrow\overline{\hat Y}=\bar Y$ |

## Exercise 3.12–3.23

见 markdown 全文：虚拟变量共线、FWL、Sherman–Morrison、$R^2$、LOO、分别回归条件等。


## 准备数据：方程 (3.49) 样本

单身亚裔男性（`race==4`, `marital==7`, `female==0`），`experience < 45`。


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

candidates = [
    Path("../../hansen/econometrics/data/cps09mar/cps09mar.xlsx"),
    Path("/home/fang/Project/zhihu-paper/p1/hansen/econometrics/data/cps09mar/cps09mar.xlsx"),
]
DATA = next(p for p in candidates if p.exists())
print("data:", DATA.resolve())

df = pd.read_excel(DATA)
df["experience"] = df["age"] - df["education"] - 6
df["lwage"] = np.log(df["earnings"] / (df["hours"] * df["week"]))
df["exp2"] = (df["experience"] ** 2) / 100

mask_349 = (
    (df["race"] == 4)
    & (df["marital"] == 7)
    & (df["female"] == 0)
    & (df["experience"] < 45)
)
sub = df.loc[mask_349].copy()
print("n =", len(sub))
sub[["education", "experience", "lwage"]].describe()


## Exercise 3.24 (a) 估计 (3.49)


In [ ]:
def ols(y, X, names=None):
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    e = y - X @ beta
    yhat = X @ beta
    sse = float(np.sum(e ** 2))
    sst = float(np.sum((y - y.mean()) ** 2))
    r2 = 1 - sse / sst
    idx = names if names is not None else list(range(len(beta)))
    return pd.Series(beta, index=idx), e, yhat, r2, sse

y = sub["lwage"].to_numpy(float)
X = np.column_stack([
    sub["education"].to_numpy(float),
    sub["experience"].to_numpy(float),
    sub["exp2"].to_numpy(float),
    np.ones(len(sub)),
])
names = ["education", "experience", "exp2/100", "intercept"]
beta, e, yhat, r2, sse = ols(y, X, names)
print(beta.to_string())
print(f"R^2 = {r2:.6f}")
print(f"SSE = {sse:.6f}")
print("Book (3.49) approx: 0.144, 0.043, -0.095, 0.531")


## Exercise 3.24 (b)(c) FWL 残差回归


In [ ]:
Z = np.column_stack([
    sub["experience"].to_numpy(float),
    sub["exp2"].to_numpy(float),
    np.ones(len(sub)),
])
edu = sub["education"].to_numpy(float)

b_y = np.linalg.lstsq(Z, y, rcond=None)[0]
b_e = np.linalg.lstsq(Z, edu, rcond=None)[0]
ry = y - Z @ b_y
re = edu - Z @ b_e

Xfwl = np.column_stack([re, np.ones(len(re))])
beta_fwl, e_fwl, _, r2_fwl, sse_fwl = ols(ry, Xfwl, ["education (FWL)", "intercept"])
print(beta_fwl.to_string())
print(f"FWL R^2 = {r2_fwl:.6f}")
print(f"FWL SSE = {sse_fwl:.6f}")
print("slope equals full OLS?", np.isclose(beta_fwl.iloc[0], beta["education"]))
print("SSE equals full OLS?", np.isclose(sse_fwl, sse))
print("R^2 equal? (expect False)", np.isclose(r2_fwl, r2))


## Exercise 3.25 数值核对 OLS 性质


In [ ]:
X1 = sub["education"].to_numpy(float)
X2 = sub["experience"].to_numpy(float)

checks = {
    "(a) sum e": np.sum(e),
    "(b) sum X1*e": np.sum(X1 * e),
    "(c) sum X2*e": np.sum(X2 * e),
    "(d) sum X1^2*e": np.sum(X1**2 * e),
    "(e) sum X2^2*e": np.sum(X2**2 * e),
    "(f) sum Yhat*e": np.sum(yhat * e),
    "(g) sum e^2": np.sum(e**2),
}
for k, v in checks.items():
    print(f"{k:20s} = {v: .6e}")

print()
print("Theory notes:")
print("- (a)(b)(c)(f) ~ 0: intercept, edu, exp in X; Yhat in col(X)")
print("- (e) ~ 0: exp^2/100 is in the regression")
print("- (d) != 0: education^2 not in the regression")
print("- (g) = SSE")


## Exercise 3.26 白人男性西班牙裔 log wage 回归

基准：Midwest；婚姻基准：never married；widowed 与 divorced 合并。


In [ ]:
mask_w = (df["race"] == 1) & (df["female"] == 0) & (df["hisp"] == 1)
s2 = df.loc[mask_w].copy()
print("n white male Hispanic =", len(s2))

s2["married"] = s2["marital"].isin([1, 2, 3]).astype(float)
s2["wid_div"] = s2["marital"].isin([4, 5]).astype(float)
s2["separated"] = (s2["marital"] == 6).astype(float)
s2["NE"] = (s2["region"] == 1).astype(float)
s2["South"] = (s2["region"] == 3).astype(float)
s2["West"] = (s2["region"] == 4).astype(float)

y2 = s2["lwage"].to_numpy(float)
X2m = np.column_stack([
    s2["education"], s2["experience"], s2["exp2"],
    s2["NE"], s2["South"], s2["West"],
    s2["married"], s2["wid_div"], s2["separated"],
    np.ones(len(s2)),
])
names2 = [
    "education", "experience", "exp2/100",
    "Northeast", "South", "West",
    "married", "widowed/divorced", "separated",
    "intercept",
]
beta2, e2, yhat2, r2_2, sse2 = ols(y2, X2m, names2)
print(beta2.to_string())
print(f"R^2 = {r2_2:.6f}, SSE = {sse2:.6f}")


### 3.26 (b) 与 statsmodels 对照


In [ ]:
try:
    import statsmodels.api as sm
    res = sm.OLS(y2, X2m).fit()
    cmp = pd.DataFrame({
        "numpy_lstsq": beta2.values,
        "statsmodels": res.params,
        "abs_diff": np.abs(beta2.values - res.params),
    }, index=names2)
    display(cmp) if "display" in dir() else print(cmp)
    print("max abs diff =", float(cmp["abs_diff"].max()))
except Exception as ex:
    print("statsmodels unavailable or error:", ex)
    print(beta2)
